# Абляции и методология

Цель — показать явно, **какой фактор отвечает за какое изменение**, и не
приписывать эффект тому, что менялось заодно. Общее правило, которому следует
вся цепочка конфигов `configs/experiments/disentangle_*`: **один YAML — одно
архитектурное решение относительно родителя**. Раздел 1 проверяет это
программно, а не на словах.

Разделение по железу:
- **Без GPU** (разделы 1-2): диффы конфигов, GFLOPs.
- **Нужен GPU + обученный чекпоинт** (разделы 3-5): зануление gate на реальных
  весах, зонды, таксономия отказов. Ячейки написаны так, чтобы **явно и без
  падения** сообщать, что чекпоинта пока нет, а не притворяться результатом.


In [ ]:
# %% Зависимости.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
               "numpy", "pandas", "pyarrow", "torch"], check=True)


In [ ]:
# %% Корень проекта.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Откройте ноутбук из корня репозитория или из notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)


## 1. Один фактор на звено цепочки — проверка по коду, не по комментарию

Раньше в этом проекте плечо (arm) `H` отвечало сразу на два вопроса
(`aux_weight` было намертво связано с `model="hf"` в устаревшей архитектуре) —
эффект приписать было нечему. Проверяем, что в текущей цепочке
`disentangle_b2_li760_r8_*` такого нет: сравниваем `to_flat_dict()` каждого
звена с его прямым родителем и печатаем **только различающиеся ключи**.


In [ ]:
# %% Диффы конфигов вдоль всей цепочки (флэт-представление, без вложенных секций).
from dataclasses import fields

from src.config import load_experiment_config

CHAIN = [
    "configs/baseline.yaml",
    "configs/experiments/disentangle_fuse.yaml",
    "configs/experiments/disentangle_fuse_cross32.yaml",
    "configs/experiments/disentangle_b2_li760_long.yaml",
    "configs/experiments/disentangle_b2_li760_r8_long.yaml",
    "configs/experiments/disentangle_b2_li760_r8_all_data_ft.yaml",
    "configs/experiments/disentangle_b2_li760_r8_all_data_hard_pixel_ft.yaml",
]

flat = [load_experiment_config(ROOT / path).to_flat_dict() for path in CHAIN]
for (parent_path, parent), (child_path, child) in zip(zip(CHAIN, flat), zip(CHAIN[1:], flat[1:])):
    changed = {k: (parent.get(k), child.get(k)) for k in child if parent.get(k) != child.get(k)}
    print(f"{Path(child_path).name}  (родитель: {Path(parent_path).name})")
    for key, (before, after) in changed.items():
        print(f"    {key}: {before!r} -> {after!r}")
    print()


**Как читать вывод (реальный, см. ячейку выше).** `run_name` и `finetune_from`
меняются почти везде — это техническая необходимость дотюна, не самостоятельный
фактор. Большинство звеньев действительно меняют один связный аспект:
`disentangle_fuse` — только уровни DG-Force и их лосс, `disentangle_fuse_cross32`
— только `disentangle_cross_strides`, `disentangle_b2_li760_r8_long` — только
`disentangle_reduction`, `..._all_data_ft` — только LR-план + переход на все
данные, `..._all_data_hard_pixel_ft` — только hard-pixel loss (плюс две GPU-
оптимизации без влияния на семантику, см. `gpu_training_optimizations.md`).

Одно звено — честно фиксируем — **не** однофакторное: `disentangle_b2_li760_long`
одновременно меняет энкодер (`pvt_v2_b2` → `pvt_v2_b2_li`), `image_size`
(640 → 760) и учебное расписание (6 → 18 эпох, +3 полных прохода). Это осознанно
бандловый шаг «переход на итоговый режим обучения» перед серией однофакторных
правок `reduction`/`all_data`/`hard_pixel` дальше по цепочке, а не нарушение
методологии по недосмотру — но именно поэтому по одному числу AIC до/после этого
шага **нельзя** сказать, какая из трёх смен дала эффект: если понадобится
изолировать вклад энкодера/разрешения/расписания по отдельности, нужен отдельный
набор промежуточных конфигов, которого сейчас в репозитории нет.


## 2. Вклад отдельных модулей в GFLOPs (без обучения)

`disentangle_mode='supervision'` не добавляет операций на инференсе (головы
существуют только для `training=True`, см. `src/modules/forensic_disentangle.py`,
`DisentangleLevel.forward`: при `not inject and not supervise` слой возвращает
вход без изменений). Проверяем это утверждение прямым замером, а не верим
комментарию в коде.


In [ ]:
# %% Supervision-режим действительно бесплатен на инференсе; fuse — нет.
import torch
from torch.utils.flop_counter import FlopCounterMode

from src.modules.forensic_disentangle import ForensicDisentangle

STRIDES = [4, 8, 16, 32]
CHANNELS = [64, 128, 320, 512]

def pyramid(size=128, batch=1):
    return [torch.randn(batch, c, size // s, size // s) for s, c in zip(STRIDES, CHANNELS)]

for mode in ("supervision", "fuse"):
    block = ForensicDisentangle(STRIDES, CHANNELS, STRIDES, mode=mode, cross_strides=[32] if mode == "fuse" else []).eval()
    counter = FlopCounterMode(display=False)
    with torch.no_grad(), counter:
        block(pyramid(), supervise=False)  # инференс: supervise=False
    print(f"mode={mode:<12} inference GFLOPs (128x128, страйды 4..32): {counter.get_total_flops() / 1e9:.6f}")


## 3. Абляционная дельта на forward-пути: зануление `channel_gate`

Модуль в forward-пути (в отличие от чисто тренировочной supervision-головы)
можно измерить абляционной дельтой **внутри одного прогона**: занулить
`channel_gate` на уже готовом чекпоинте и сделать второй проход по
development — без переобучения. `channel_gate` инициализируется нулём (см.
`DisentangleLevel.__init__`), поэтому необученный арм побитово равен базовой
модели; зануление на обученных весах откатывает именно вклад резидуала, не
архитектуру целиком.

Аналогично можно занулить гейты `ForensicFusion.fusion_blocks['8'|'16'|'32']`
— это ablation вклада JPEG/DCT-ветки в конечный fusion (сама JPEG-ветка
продолжает считаться, но её результат обнуляется перед добавлением к RGB-
признакам).


In [ ]:
# %% Абляция channel_gate на реальном чекпоинте — выполняется, если чекпоинт есть.
import copy

from src.config import load_experiment_config
from src.training.builders import build_model
from src.training.runs import Run

FINAL_RUN = ROOT / "runs" / "disentangle_b2_li760_r8_all_data_hard_pixel_ft"
checkpoint_path = FINAL_RUN / "ckpt" / "last.pt"

if not checkpoint_path.is_file():
    print(f"Пропуск: чекпоинт не найден ({checkpoint_path}). "
          f"Этот блок нужно перезапустить после обучения (solution.ipynb, раздел 7).")
else:
    run = Run.open(FINAL_RUN)
    cfg = load_experiment_config(ROOT / "configs/experiments/disentangle_b2_li760_r8_all_data_hard_pixel_ft.yaml")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    weights_key = "ema" if checkpoint.get("ema") is not None else "model"

    def load_model():
        model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False)
        model.load_state_dict(checkpoint[weights_key])
        return model.eval()

    full_model = load_model()
    ablated_model = load_model()
    with torch.no_grad():
        for block in ablated_model.disentangle.blocks.values():
            if block.inject:
                block.channel_gate.zero_()
        for cross_block in ablated_model.disentangle.cross.values():
            cross_block.channel_gate.zero_()
        for fusion_block in ablated_model.forensic_fusion.fusion_blocks.values():
            fusion_block.channel_gate.zero_()

    print("Полная и абляционная модели построены — сравнение Dice по development см. ниже (нужны данные).")


**TODO (авторам):** для честной абляционной дельты нужен парный прогон по
`development` (не по одной картинке): взять `val_loader` из `src.training.builders`
или `runs/<run>/development/per_image.parquet`, прогнать обе модели
(`full_model`, `ablated_model`) на одних и тех же изображениях и сравнить Dice
**по каждой картинке** (`scipy.stats.bootstrap` или ручной paired bootstrap по
разностям) — точечная разница по одной итоговой цифре здесь не значима: между
двумя идентичными повторными прогонами AIC и так колеблется на уровне
нескольких тысячных, и без парного сравнения нельзя отличить эффект абляции от
шума инициализации/порядка данных. Впишите фактический σ, посчитанный на вашем
протоколе, если он отличается от типичных оценок такого масштаба.


## 4. Линейные зонды (по признакам encoder, до decoder/DG-Force)

Быстрая, дёшево считаемая диагностика: обучить лёгкий линейный классификатор
поверх усреднённых по пространству признаков страйдов 16 и 32 (`encoder`,
до `ForensicDisentangle`) предсказывать `is_negative`/domain — если зонд уже
неплохо разделяет классы на замороженных признаках энкодера, часть работы,
которую выполняет decoder/DG-Force, может быть избыточной для этой конкретной
подзадачи (но не для точной сегментации границы — зонд видит только глобальный
пуллинг). Обёрнуто так же безопасно: без чекпоинта — печатает предупреждение,
не падает.


In [ ]:
# %% Линейные зонды поверх замороженных признаков энкодера (нужен чекпоинт + данные).
from sklearn.linear_model import LogisticRegression  # тот же sklearn, что уже в environment.yml

if not checkpoint_path.is_file():
    print("Пропуск: нужен обученный чекпоинт (см. раздел 3).")
else:
    print("Каркас готов: extract pooled features on strides 16/32 from `full_model.encoder`, "
          "затем LogisticRegression(max_iter=1000).fit(features, labels) с train/val сплитом "
          "по protocol-ролям (не по случайному разбиению, чтобы не потерять group leakage guard).")


## 5. Таксономия отказов по `development/per_image.parquet`

Валидация текущего пайплайна уже пишет пофреймовые метрики
(`runs/<run>/development/per_image.parquet`, см. `README.md`, раздел
«Оценка и submission»). Раскладываем по простым, объяснимым категориям
(не ML-моделью, а порогами по Dice/площади) — это дешёвая, интерпретируемая
диагностика, которая не требует повторного инференса.


In [ ]:
# %% Таксономия отказов по сохранённым per-image метрикам стадии A (development).
import pandas as pd

per_image_path = ROOT / "runs" / "disentangle_b2_li760_r8_long" / "development" / "per_image.parquet"

if not per_image_path.is_file():
    print(f"Пропуск: {per_image_path} появится после обучения стадии A (solution.ipynb, раздел 5).")
else:
    per_image = pd.read_parquet(per_image_path)

    def classify(row) -> str:
        if row.get("is_negative"):
            return "ok (негатив)" if not row.get("false_positive", False) else "ложная тревога"
        dice = row.get("dice", float("nan"))
        if pd.isna(dice) or dice == 0:
            return "слепа (Dice=0)"
        if dice < 0.3:
            return "мимо (Dice<0.3)"
        if dice < 0.7:
            return "частично (0.3<=Dice<0.7)"
        return "точно (Dice>=0.7)"

    per_image["failure_bucket"] = per_image.apply(classify, axis=1)
    taxonomy = per_image["failure_bucket"].value_counts(normalize=True).rename("share")
    taxonomy


**TODO (авторам):** колонки `per_image.parquet` нужно свести с фактической
схемой, которую пишет `src/eval/protocol.py`/`src/training/validation.py` на
вашем прогоне (здесь предполагаются `is_negative`, `false_positive`, `dice` —
проверьте по факту после первого запуска `solution.ipynb`). Отдельно стоит
посчитать долю `plain_l9` (см. `notebooks/eda.ipynb`, раздел 1) внутри каждой
корзины — по прежним наблюдениям именно на этом под-домене недобор Dice
особенно заметен, но эта версия ноутбука не подтверждает цифру без реального
прогона.


## 6. Hard-pixel loss (стадия C): диагностика по кривой обучения, не A/B

У стадии C нет контролируемого A/B (парного прогона с/без `hard_pixel_weight`
на идентичных данных) — она "слепой" финальный дотюн (см. `solution.ipynb`,
раздел 7). Здесь — не заявка на измеренный прирост AIC, а честная проверка,
что компонент лосса действительно обучается (убывает), а не игнорируется
оптимизатором.


In [ ]:
# %% Кривая hard_pixel_bce по логам стадии C (если она уже обучалась).
metrics_path = ROOT / "runs" / "disentangle_b2_li760_r8_all_data_hard_pixel_ft" / "metrics.csv"

if not metrics_path.is_file():
    print(f"Пропуск: {metrics_path} появится во время/после обучения стадии C.")
else:
    metrics = pd.read_csv(metrics_path)
    hard_pixel_columns = [c for c in metrics.columns if "hard_pixel" in c]
    print("Найденные колонки hard-pixel лосса:", hard_pixel_columns)
    metrics[["step", *hard_pixel_columns]].tail(20) if "step" in metrics.columns else metrics[hard_pixel_columns].tail(20)
